# TESS Candidate Transit Analysis — TIC 441420236

This notebook completes the internship assignment: read a TESS light-curve FITS file, plot the light curve, and zoom in on one candidate transit. I also added quality-flag checks, a local detrend, and an aperture-mask visualization.

**Selected transit center:** TBJD `1330.402640`  
**Target:** `TIC 441420236`  
**Sector:** `1`


## 1. Setup

The lecture notebook used `astropy.io.fits`. This version uses `src/fits_reader.py`, a small FITS binary-table reader included with the repo, so the project can run even in lightweight Python environments. It also demonstrates how the FITS header and binary table structure translate into arrays.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from fits_reader import list_hdus, read_bintable, read_image

LC_FILE = ROOT / "data" / "raw" / "tess2018206045859-s0001-0000000441420236-0120-s_lc.fits"
DVT_FILE = ROOT / "data" / "raw" / "tess2018206190142-s0001-s0001-0000000441420236-00366_dvt.fits"

## 2. Read the FITS light curve

The first extension of the TESS light-curve FITS file is a binary table. Important columns include:

- `TIME`: timestamp in TBJD
- `PDCSAP_FLUX`: systematics-corrected flux
- `PDCSAP_FLUX_ERR`: flux uncertainty
- `QUALITY`: cadence quality flag


In [ ]:
lc, lc_header = read_bintable(LC_FILE, hdu_index=1)

for col in lc.dtype.names:
    print(col)

print("
Target:", lc_header["OBJECT"].strip())
print("Sector:", lc_header["SECTOR"])
print("Camera / CCD:", lc_header["CAMERA"], lc_header["CCD"])

In [ ]:
time = lc["TIME"].astype(float)
flux = lc["PDCSAP_FLUX"].astype(float)
flux_err = lc["PDCSAP_FLUX_ERR"].astype(float)
quality = lc["QUALITY"].astype(int)

finite = np.isfinite(time) & np.isfinite(flux)
good = finite & (quality == 0)
flagged_finite = finite & (quality != 0)

median_flux = np.nanmedian(flux[good])
normalized_flux = flux / median_flux
normalized_err = flux_err / median_flux

print(f"Total cadences: {len(time):,}")
print(f"Finite PDCSAP cadences: {finite.sum():,}")
print(f"Finite and QUALITY == 0: {good.sum():,}")
print(f"Finite but flagged: {flagged_finite.sum():,}")
print(f"Median PDCSAP flux: {median_flux:,.2f} e-/s")

## 3. Full light curve

This plot gives context for the target's variability over the whole observing sector. The shaded region marks the transit window I zoom into for the required assignment plot.

In [ ]:
dvt_hdus = list_hdus(DVT_FILE)
tce5_header = dvt_hdus[5]["header"]

t0 = float(tce5_header["TEPOCH"])
period_days = float(tce5_header["TPERIOD"])
duration_hours = float(tce5_header["TDUR"])
duration_days = duration_hours / 24
reported_depth_ppm = float(tce5_header["TDEPTH"])

fig, ax = plt.subplots(figsize=(11, 5.6))
ax.scatter(time[good], normalized_flux[good], s=6, alpha=0.55, label="quality = 0")
ax.scatter(time[flagged_finite], normalized_flux[flagged_finite], s=12, alpha=0.85, label="finite but flagged")
ax.axvspan(t0 - 0.25, t0 + 0.25, alpha=0.18, label="zoomed transit window")
ax.axvline(t0, linestyle="--", linewidth=1.2, label=f"TCE epoch = {t0:.3f} TBJD")
ax.set_title(f"TESS Sector {lc_header['SECTOR']} Light Curve for {lc_header['OBJECT'].strip()}")
ax.set_xlabel("Time (TBJD = BJD - 2457000)")
ax.set_ylabel("PDCSAP Flux / Median")
ax.set_ylim(0.982, 1.055)
ax.legend(loc="upper right", frameon=True)
plt.show()

## 4. Required zoomed transit plot

This is the most important plot for the assignment: I changed the time and flux ranges to show one candidate planet transit.

In [ ]:
dt = time - t0
zoom = good & (np.abs(dt) < 0.35)
oot_for_fit = zoom & (np.abs(dt) > duration_days * 0.85) & (np.abs(dt) < 0.32)
coeffs = np.polyfit(dt[oot_for_fit], normalized_flux[oot_for_fit], deg=2)
local_baseline = np.polyval(coeffs, dt)
detrended = normalized_flux / local_baseline

in_transit = zoom & (np.abs(dt) <= duration_days / 2)
out_of_transit = zoom & (np.abs(dt) > duration_days * 1.2) & (np.abs(dt) < 0.28)
estimated_depth_ppm = (1 - np.nanmedian(detrended[in_transit])) * 1_000_000
local_scatter_ppm = np.nanstd(detrended[out_of_transit]) * 1_000_000

fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.errorbar(time[zoom], normalized_flux[zoom], yerr=normalized_err[zoom], fmt=".", markersize=3.2, elinewidth=0.45, alpha=0.72, label="PDCSAP flux")
dense_t = np.linspace(-0.25, 0.25, 300)
ax.plot(t0 + dense_t, np.polyval(coeffs, dense_t), linewidth=2.0, label="local stellar baseline")
ax.axvline(t0, linestyle="--", linewidth=1.1, label="candidate transit center")
ax.axvspan(t0 - duration_days / 2, t0 + duration_days / 2, alpha=0.18, label=f"DVT duration ≈ {duration_hours:.2f} hr")
ax.set_xlim(t0 - 0.25, t0 + 0.25)
ax.set_ylim(0.989, 1.003)
ax.set_title(f"Required Transit Zoom: {lc_header['OBJECT'].strip()}, TESS Sector {lc_header['SECTOR']}")
ax.set_xlabel("Time (TBJD)")
ax.set_ylabel("Median-normalized PDCSAP Flux")
ax.text(0.02, 0.04, f"Estimated local depth ≈ {estimated_depth_ppm:,.0f} ppm", transform=ax.transAxes, fontsize=10,
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.82, "edgecolor": "0.8"})
ax.legend(loc="lower left", frameon=True)
plt.show()

## 5. Local detrend view

The star is variable, so a local baseline helps isolate the short transit-like dip from slower stellar variability.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.scatter((time[zoom] - t0) * 24, detrended[zoom], s=12, alpha=0.75, label="locally detrended flux")
ax.axhline(1.0, linestyle="--", linewidth=1.0)
ax.axvline(0.0, linestyle="--", linewidth=1.0, label="mid-transit")
ax.axvspan(-duration_hours / 2, duration_hours / 2, alpha=0.18, label="DVT duration")
ax.set_xlim(-6, 6)
ax.set_ylim(0.9955, 1.0025)
ax.set_title("Local Detrend of the Candidate Transit")
ax.set_xlabel("Hours from selected transit center")
ax.set_ylabel("Flux / Local Polynomial Baseline")
ax.text(0.02, 0.05, f"Depth ≈ {estimated_depth_ppm:,.0f} ppm
OOT scatter ≈ {local_scatter_ppm:,.0f} ppm", transform=ax.transAxes, fontsize=10,
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.82, "edgecolor": "0.8"})
ax.legend(loc="upper right", frameon=True)
plt.show()

## 6. Aperture mask

The second extension of the light-curve FITS file is an image of the photometric aperture bitmask. This helps show which pixels contributed to the light curve.

In [ ]:
aperture, aperture_header = read_image(LC_FILE, hdu_index=2)
fig, ax = plt.subplots(figsize=(6.4, 6.0))
im = ax.imshow(aperture, origin="lower")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Aperture bitmask value")
ax.set_title(f"TESS Aperture Mask for {lc_header['OBJECT'].strip()}")
ax.set_xlabel("Column pixel")
ax.set_ylabel("Row pixel")
ax.text(0.03, 0.97, f"NPIXSAP = {aperture_header.get('NPIXSAP', 'unknown')}", transform=ax.transAxes, va="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.82, "edgecolor": "0.8"})
plt.show()

## 7. Summary

- The required zoomed plot shows the candidate event around **TBJD 1330.402640**.
- The FITS table contains **20,076** cadences, with **18,050** finite clean cadences after selecting `QUALITY == 0`.
- My simple local depth estimate is **2,779 ppm**.
- The DVT header depth is **3,024 ppm**.
- The final repository includes scripts, raw data, processed summary tables, and plots for GitHub/Discord.
